# Tyre Method Refinement

This notebook compares a few ways to estimate `T_0` and degradation `d` across multiple races/drivers.

Goal: find a method that is more stable than the current per-compound split, and check when monotonic compound ordering breaks.


In [5]:
import os
from typing import Dict, List, Tuple

import fastf1
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import theilslopes

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 80)

os.makedirs('fastf1_cache', exist_ok=True)
fastf1.Cache.enable_cache('fastf1_cache')

RACE_SPECS = [
    (2025, 'Abu Dhabi', 'R'),
    (2025, 'Bahrain', 'R'),
    (2025, 'Monaco', 'R'),
]

COMPOUND_ORDER = ['SOFT', 'MEDIUM', 'HARD']
MIN_STINT_LAPS = 6
FUEL_FRONTIER_Q = 0.20


In [6]:
def load_race_laps(year: int, grand_prix: str, session_name: str = 'R'):
    session = fastf1.get_session(year, grand_prix, session_name)
    session.load()
    return session, session.laps.copy()


def prepare_nonpit_laps(laps_df: pd.DataFrame, group_cols: List[str]) -> pd.DataFrame:
    laps = laps_df.copy()
    laps = laps[laps['LapTime'].notna()].copy()
    laps['CompoundU'] = laps['Compound'].str.upper()
    laps['LapTimeSec'] = laps['LapTime'].dt.total_seconds()
    laps = laps[laps['PitInTime'].isna() & laps['PitOutTime'].isna()].copy()
    laps = laps.pick_quicklaps(threshold=1.07)
    laps = laps.sort_values(group_cols + ['LapNumber']).copy()
    laps['TyreLap'] = laps.groupby(group_cols).cumcount() + 1
    laps['LapDeltaSec'] = laps.groupby(group_cols)['LapTimeSec'].diff()
    return laps


def estimate_fuel_track_gain_from_field(session_laps: pd.DataFrame, frontier_quantile: float = FUEL_FRONTIER_Q) -> float:
    field_nonpit = prepare_nonpit_laps(session_laps, group_cols=['Driver', 'Stint'])
    frontier = (
        field_nonpit.groupby('LapNumber')['LapTimeSec']
        .quantile(frontier_quantile)
        .dropna()
        .sort_index()
    )
    deltas = frontier.diff().dropna()
    if len(deltas) < 5:
        raise ValueError('Not enough field-lap deltas to estimate fuel/track gain.')
    return float(np.median(deltas))


def trim_adjusted_laps(tmp: pd.DataFrame, trim_mode: str = 'iqr', trim_strength: float = 1.5) -> pd.DataFrame:
    if trim_mode == 'none':
        return tmp.copy()

    values = tmp['AdjLapTime'].to_numpy(dtype=float)
    if trim_mode == 'iqr':
        q1, q3 = np.quantile(values, [0.25, 0.75])
        iqr = q3 - q1
        low, high = q1 - trim_strength * iqr, q3 + trim_strength * iqr
        return tmp[(tmp['AdjLapTime'] >= low) & (tmp['AdjLapTime'] <= high)].copy()

    if trim_mode == 'mad':
        med = float(np.median(values))
        mad = float(np.median(np.abs(values - med)))
        if mad == 0:
            return tmp.copy()
        robust_z = 0.6745 * (tmp['AdjLapTime'] - med) / mad
        return tmp[np.abs(robust_z) <= trim_strength].copy()

    raise ValueError(f'Unknown trim_mode: {trim_mode}')


def fit_theil_sen(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or np.ptp(x) == 0:
        return np.nan, np.nan, np.nan
    slope, intercept, _, _ = theilslopes(y, x)
    return float(slope), float(intercept), float(intercept + slope)


def pava(values, increasing=True):
    y = np.asarray(values, dtype=float)
    if not increasing:
        y = -y
    blocks = [[float(v), 1] for v in y]
    i = 0
    while i < len(blocks) - 1:
        if blocks[i][0] <= blocks[i + 1][0]:
            i += 1
            continue
        total = blocks[i][0] * blocks[i][1] + blocks[i + 1][0] * blocks[i + 1][1]
        count = blocks[i][1] + blocks[i + 1][1]
        merged = [total / count, count]
        blocks[i : i + 2] = [merged]
        i = max(i - 1, 0)
    out = []
    for value, count in blocks:
        out.extend([value] * count)
    out = np.asarray(out, dtype=float)
    return out if increasing else -out


def order_violation(values_by_compound: Dict[str, float], expected_order: List[str], increasing=True) -> bool:
    seq = [values_by_compound[c] for c in expected_order if c in values_by_compound]
    if len(seq) < 2:
        return False
    pairs = zip(seq, seq[1:])
    return any((b < a) if increasing else (b > a) for a, b in pairs)


In [7]:
def estimate_compound_methods(nonpit_laps: pd.DataFrame, compounds: List[str], fuel_track_gain: float, trim_mode: str = 'iqr', trim_strength: float = 1.5) -> pd.DataFrame:
    rows = []
    for compound in compounds:
        tmp = nonpit_laps[nonpit_laps['CompoundU'] == compound].copy()
        if len(tmp) < 8:
            continue

        tmp['AdjLapTime'] = tmp['LapTimeSec'] - fuel_track_gain * (tmp['LapNumber'] - 1)
        tmp = trim_adjusted_laps(tmp, trim_mode=trim_mode, trim_strength=trim_strength)
        if len(tmp) < 6:
            continue

        fresh_laps = tmp[tmp['TyreLap'] <= 2]['LapTimeSec']
        if fresh_laps.empty:
            fresh_laps = tmp['LapTimeSec']
        t0_fresh = float(fresh_laps.median())

        pooled_d, pooled_b, pooled_t0 = fit_theil_sen(tmp['TyreLap'], tmp['AdjLapTime'])

        stint_t0 = []
        stint_d = []
        for (_, stint_df) in tmp.groupby(['Driver', 'Stint']):
            if len(stint_df) < MIN_STINT_LAPS:
                continue
            d, b, t0 = fit_theil_sen(stint_df['TyreLap'], stint_df['AdjLapTime'])
            if np.isnan(d) or np.isnan(t0):
                continue
            stint_d.append(d)
            stint_t0.append(t0)

        t0_stint = float(np.median(stint_t0)) if stint_t0 else pooled_t0
        d_stint = float(np.median(stint_d)) if stint_d else pooled_d

        deltas = tmp.groupby(['Driver', 'Stint'])['AdjLapTime'].diff().dropna()
        d_delta = float(np.median(deltas)) if len(deltas) else np.nan

        rows.append({
            'compound': compound,
            'fresh_t0': t0_fresh,
            'pooled_t0': pooled_t0,
            'stint_t0': t0_stint,
            'pooled_d': pooled_d,
            'stint_d': d_stint,
            'delta_d': d_delta,
            'sample_count': len(tmp),
            'stint_count': int(tmp.groupby(['Driver', 'Stint']).ngroups),
        })

    return pd.DataFrame(rows).set_index('compound').sort_index()


def summarize_method_frame(method_frame: pd.DataFrame, method_name: str) -> Dict:
    t0_values = method_frame['t0'].to_dict()
    d_values = method_frame['d'].to_dict()
    t0_viol = order_violation(t0_values, COMPOUND_ORDER, increasing=True)
    d_viol = order_violation(d_values, COMPOUND_ORDER, increasing=False)
    t0_iso = pava([t0_values[c] for c in COMPOUND_ORDER], increasing=True)
    d_iso = pava([d_values[c] for c in COMPOUND_ORDER], increasing=False)
    return {
        'method': method_name,
        't0_violation': t0_viol,
        'd_violation': d_viol,
        't0_adjustment_mean_abs': float(np.mean(np.abs(t0_iso - np.array([t0_values[c] for c in COMPOUND_ORDER])))),
        'd_adjustment_mean_abs': float(np.mean(np.abs(d_iso - np.array([d_values[c] for c in COMPOUND_ORDER])))),
    }


In [ ]:
session_rows = []
method_rows = []

for year, grand_prix, session_name in RACE_SPECS:
    session, laps = load_race_laps(year, grand_prix, session_name)
    nonpit = prepare_nonpit_laps(laps, group_cols=['Driver', 'Stint'])
    nonpit = nonpit[nonpit['CompoundU'].isin(COMPOUND_ORDER)].copy()
    if nonpit.empty:
        raise ValueError(f'No clean laps for {year} {grand_prix}')

    fuel_gain = estimate_fuel_track_gain_from_field(laps)
    compound_methods = estimate_compound_methods(nonpit, COMPOUND_ORDER, fuel_gain)

    for method_name, col_map in {
        'fresh_median+pooled_ts': ('fresh_t0', 'pooled_d'),
        'pooled_ts': ('pooled_t0', 'pooled_d'),
        'stint_median_ts': ('stint_t0', 'stint_d'),
        'delta_only': ('fresh_t0', 'delta_d'),
    }.items():
        raw_frame = pd.DataFrame({
            't0': compound_methods[col_map[0]],
            'd': compound_methods[col_map[1]],
        }).reindex(COMPOUND_ORDER)
        iso_frame = pd.DataFrame({
            't0': pava(raw_frame['t0'].to_numpy(), increasing=True),
            'd': pava(raw_frame['d'].to_numpy(), increasing=False),
        }, index=COMPOUND_ORDER)

        for variant_name, method_frame in {
            method_name: raw_frame,
            f'{method_name}+iso': iso_frame,
        }.items():
            diag = summarize_method_frame(method_frame, variant_name)
            diag['t0_repair_mean_abs'] = float(np.mean(np.abs(method_frame['t0'].to_numpy() - raw_frame['t0'].to_numpy())))
            diag['d_repair_mean_abs'] = float(np.mean(np.abs(method_frame['d'].to_numpy() - raw_frame['d'].to_numpy())))
            diag.update({'year': year, 'grand_prix': grand_prix, 'session': session_name})
            method_rows.append(diag)

            for compound in method_frame.index:
                session_rows.append({
                    'year': year,
                    'grand_prix': grand_prix,
                    'session': session_name,
                    'method': variant_name,
                    'compound': compound,
                    't0': float(method_frame.loc[compound, 't0']),
                    'd': float(method_frame.loc[compound, 'd']),
                    'fuel_track_gain': fuel_gain,
                    'sample_count': int(compound_methods.loc[compound, 'sample_count']),
                    'stint_count': int(compound_methods.loc[compound, 'stint_count']),
                })

session_results = pd.DataFrame(session_rows)
method_summary = pd.DataFrame(method_rows)

display(session_results.head(12))
display(method_summary.groupby('method')[['t0_violation', 'd_violation', 't0_adjustment_mean_abs', 'd_adjustment_mean_abs', 't0_repair_mean_abs', 'd_repair_mean_abs']].mean().sort_values(['t0_violation', 'd_violation']))


In [ ]:
example = session_results[(session_results['grand_prix'] == RACE_SPECS[0][1]) & (session_results['method'] == 'stint_median_ts')].copy()
if not example.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
    example.pivot(index='compound', columns='method', values='t0').reindex(COMPOUND_ORDER).plot(kind='bar', ax=axes[0])
    axes[0].set_title('T0 by compound')
    axes[0].set_ylabel('sec')
    example.pivot(index='compound', columns='method', values='d').reindex(COMPOUND_ORDER).plot(kind='bar', ax=axes[1], color=['#4C72B0'])
    axes[1].set_title('d by compound')
    axes[1].set_ylabel('sec/lap')
    plt.tight_layout()
    plt.show()


In [ ]:
TRIM_POLICIES = {
    'trim_none': ('none', 1.5),
    'trim_iqr_1p5': ('iqr', 1.5),
    'trim_iqr_1p0': ('iqr', 1.0),
    'trim_mad_3p5': ('mad', 3.5),
}

trim_rows = []
for year, grand_prix, session_name in RACE_SPECS:
    session, laps = load_race_laps(year, grand_prix, session_name)
    nonpit = prepare_nonpit_laps(laps, group_cols=['Driver', 'Stint'])
    nonpit = nonpit[nonpit['CompoundU'].isin(COMPOUND_ORDER)].copy()
    fuel_gain = estimate_fuel_track_gain_from_field(laps)

    for trim_name, (trim_mode, trim_strength) in TRIM_POLICIES.items():
        compound_methods = estimate_compound_methods(
            nonpit,
            COMPOUND_ORDER,
            fuel_gain,
            trim_mode=trim_mode,
            trim_strength=trim_strength,
        )
        raw_frame = pd.DataFrame({
            't0': compound_methods['fresh_t0'],
            'd': compound_methods['pooled_d'],
        }).reindex(COMPOUND_ORDER)
        iso_frame = pd.DataFrame({
            't0': pava(raw_frame['t0'].to_numpy(), increasing=True),
            'd': pava(raw_frame['d'].to_numpy(), increasing=False),
        }, index=COMPOUND_ORDER)

        for variant_name, method_frame in {
            'fresh_median+pooled_ts': raw_frame,
            'fresh_median+pooled_ts+iso': iso_frame,
        }.items():
            trim_rows.append({
                'year': year,
                'grand_prix': grand_prix,
                'session': session_name,
                'trim_policy': trim_name,
                'method': variant_name,
                't0_violation': order_violation(method_frame['t0'].to_dict(), COMPOUND_ORDER, increasing=True),
                'd_violation': order_violation(method_frame['d'].to_dict(), COMPOUND_ORDER, increasing=False),
                't0_repair_mean_abs': float(np.mean(np.abs(method_frame['t0'].to_numpy() - raw_frame['t0'].to_numpy()))),
                'd_repair_mean_abs': float(np.mean(np.abs(method_frame['d'].to_numpy() - raw_frame['d'].to_numpy()))),
                'sample_count_soft': int(compound_methods.loc['SOFT', 'sample_count']) if 'SOFT' in compound_methods.index else np.nan,
            })

trim_summary = pd.DataFrame(trim_rows)
display(trim_summary.groupby(['trim_policy', 'method'])[['t0_violation', 'd_violation', 't0_repair_mean_abs', 'd_repair_mean_abs']].mean().sort_values(['t0_violation', 'd_violation', 't0_repair_mean_abs']))
display(trim_summary.groupby('trim_policy')[['sample_count_soft']].mean().sort_values('sample_count_soft'))


## How to read this

- `fresh_median+pooled_ts` is closest to the current notebook logic.
- `pooled_ts` ties `T_0` and `d` to the same robust line fit.
- `stint_median_ts` is the main candidate if the pooled fit is still too noisy.
- `delta_only` is a sanity check, not usually the final choice.

The `+iso` variants show the monotonic repair directly. Recommended starting point: `fresh_median+pooled_ts+iso`, because it stays closest to the current model while fixing ordering with a small repair.

Extra outlier trimming mostly changes the numbers a little; it does not reliably fix compound ordering on its own, and too much trimming can make the soft sample too small.
